In [1]:
!pip install --no-index --find-links=/kaggle/input/ariel-2024-pqdm pqdm > /dev/null

In [2]:
import pandas as pd
import numpy as np
import torch
import torch.nn.functional as F
import multiprocessing as mp
import torch.nn as nn
import os
import matplotlib.pyplot as plt
import itertools

from tqdm import tqdm
from pqdm.threads import pqdm
from astropy.stats import sigma_clip
from scipy.optimize import minimize
from torch.utils.data import DataLoader, TensorDataset, random_split
from sklearn.preprocessing import StandardScaler
from scipy.signal import savgol_filter
from sklearn.metrics import mean_squared_error

In [3]:
ROOT_PATH = "/kaggle/input/ariel-data-challenge-2025"
MODE = "test"

class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"

    SCALE = 0.946
    SIGMA = 0.00056
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    
    if g1_max != 0:
        g1 /= g1_max
    if g2_max != 0:
        g2 /= g2_max
    
    phase1 = int(np.argmin(g1))
    phase2 = int(np.argmax(g2)) + min_idx
    
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    
    for single in preprocessed_data:
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        fgs = single[:, 0]
        oot = (fgs[: p1 - delta] if p1 - delta > 0 else np.empty(0, fgs.dtype))
        if p2 + delta < fgs.size:
            oot = np.concatenate([oot, fgs[p2 + delta :]])
        inn = fgs[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    
    k = np.clip(k, 0.85, 1.30) 
    
    return k * cfg.SIGMA * 1.04

def estimate_sigma_air(preprocessed_data, cfg):
    """Return sigma_air length N_planets."""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        white = np.nanmean(single[:, 1:], axis=1)
        white_s = savgol_filter(white, 20, 2)

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    
    k = np.clip(k, 0.92, 1.22)

    return k * cfg.SIGMA * 1.04


class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)

    def _apply_linear_corr(self, linear_corr, signal):
        coeffs = np.flip(linear_corr, axis=0)
        x = signal.astype(np.float64, copy=False)
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)
            out += coeffs[k]

        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]

        signal = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet").to_numpy()
        dark = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet").to_numpy()
        dead = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet").to_numpy()
        flat = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet").to_numpy()
        linear_corr = pd.read_parquet(f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet").values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])

        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset

        hot = sigma_clip(dark, sigma=5, maxiters=5).mask

        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]


        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]

        np.maximum(signal, 0, out=signal)

        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)

        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment

        signal[::2] -= dark * even_scale
        signal[1::2] -= dark * odd_scale

        
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        mean_signal = np.nanmean(signal_roi, axis=1)

        cds_signal = mean_signal[1::2] - mean_signal[0::2]

        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)
            np.clip(binned, q_lo, q_hi, out=binned)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)
            med = np.nanmedian(var)

            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var

            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]


        return binned

    def _process_planet_sensor(self, args):
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def process_all_data(self):
        args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
        preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
        preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        preprocessed_signal = np.concatenate(
            [np.stack(preprocessed_fgs1), np.stack(preprocessed_airs_ch0)], axis=2
        )
        
        return preprocessed_signal
    

class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index

        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        
        return error

    def predict(self, single_preprocessed_signal):
        signal_1d = single_preprocessed_signal[:, 1:].mean(axis=1)
        signal_1d = savgol_filter(signal_1d, 20, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        phase2 = min(len(signal_1d) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)    

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
        ]
        return np.array(predictions) * self.cfg.SCALE
    
class SubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions1, predictions2, predictions, sigma_fgs=None, sigma_air=None):
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        submission_df.iloc[:, 1:283] = predictions2
        submission_df.iloc[:, 0] = predictions1

        submission_df.to_csv("submission_0.csv")
        return submission_df



config = Config()
    
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00,  5.85it/s]


In [4]:
processor = SignalProcessor(Config)
StarInfo = pd.read_csv(ROOT_PATH + f"/{MODE}_star_info.csv")
StarInfo["planet_id"] = StarInfo["planet_id"].astype(int)
PlanetIds = StarInfo["planet_id"].tolist()
StarInfo = StarInfo.set_index("planet_id")
StarInfo

,Rs,Ms,Ts,Mp,e,P,sma,i
planet_id,,,,,,,,
1103775,0.965432,0.9591,5539.03037,1.665007,0.0,6.932871,15.43293,89.533139


In [5]:
predictions = pd.DataFrame(predictions)
predictions = predictions.rename(columns={0: "transit_depth"})
predictions

,transit_depth
0,0.01585


In [6]:
input_df = StarInfo.copy()
pred_series = predictions["transit_depth"].set_axis(input_df.index, copy=False)

input_df.insert(0, "transit_depth", (pred_series * 10000).to_numpy())

features = ["transit_depth", "Rs", "i"]
X = input_df[features].values.astype("float32")

In [7]:
X

array([[158.5023  ,   0.965432,  89.53314 ]], dtype=float32)

In [8]:
class ResidualBlock(nn.Module):
    def __init__(self, dim, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return self.relu(out + identity)


class ResNetMLP(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=32, output_dim=1, num_blocks=3, dropout_rate=0.2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_dim, p=dropout_rate) for _ in range(num_blocks)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.output_layer(x)
        return x
model = ResNetMLP(num_blocks=80, dropout_rate=0.2)
model.load_state_dict(torch.load("/kaggle/input/fgs1/pytorch/default/1/best_model.pth"))
model.eval()
X_tensor = torch.tensor(X, dtype=torch.float32)
with torch.no_grad():
    predictions1 = model(X_tensor).numpy()
predictions1 /= 10000

In [9]:
predictions1

array([[0.01579722]], dtype=float32)

In [10]:
class ResidualBlock2(nn.Module):
    def __init__(self, dim, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.bn1 = nn.BatchNorm1d(dim)
        self.fc2 = nn.Linear(dim, dim)
        self.bn2 = nn.BatchNorm1d(dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        identity = x
        out = self.relu(self.bn1(self.fc1(x)))
        out = self.dropout(out)
        out = self.bn2(self.fc2(out))
        return self.relu(out + identity)


class ResNetMLP2(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, output_dim = 282, num_blocks=3, dropout_rate=0.2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[ResidualBlock(hidden_dim, p=dropout_rate) for _ in range(num_blocks)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.output_layer(x)
        return x

In [11]:
model2 = ResNetMLP2(num_blocks=80, dropout_rate=0.3)
model2.load_state_dict(torch.load("/kaggle/input/airs/pytorch/default/1/best_model_airs.pth"))
model2.eval()

with torch.no_grad():
    predictions2 = model2(X_tensor).numpy()
predictions2 /= 10000

In [12]:
predictions2

array([[0.01565252, 0.01581958, 0.01571437, 0.01570693, 0.01563848,
        0.0157747 , 0.01577698, 0.01591331, 0.01573555, 0.01574312,
        0.01578924, 0.01582134, 0.01573465, 0.01583501, 0.01574529,
        0.0156511 , 0.01585521, 0.01553765, 0.01566137, 0.01572843,
        0.01571909, 0.0159245 , 0.01580648, 0.01564254, 0.01572073,
        0.01566341, 0.01574577, 0.01580644, 0.01562952, 0.01552686,
        0.01567001, 0.01563644, 0.01596393, 0.01560699, 0.01573448,
        0.01577198, 0.01581255, 0.01568514, 0.01563672, 0.01603777,
        0.01568849, 0.0157814 , 0.01586783, 0.01567172, 0.01588679,
        0.01576289, 0.01597679, 0.0158401 , 0.01571649, 0.01574179,
        0.01586217, 0.01580781, 0.01594646, 0.01547061, 0.0158231 ,
        0.01597239, 0.01573613, 0.01578529, 0.01586087, 0.01580358,
        0.0158097 , 0.01556133, 0.01590083, 0.01588867, 0.01598836,
        0.01569308, 0.01597944, 0.01581585, 0.0157266 , 0.01603748,
        0.01600512, 0.01584975, 0.01586535, 0.01

In [13]:
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)


submission_generator = SubmissionGenerator(config)
submission = submission_generator.create(predictions1, predictions2, predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)
submission

,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,wl_10,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
planet_id,,,,,,,,,,,,,,,,,,,,,
1103775,0.015797,0.015653,0.01582,0.015714,0.015707,0.015638,0.015775,0.015777,0.015913,0.015736,...,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582,0.000582


# Model 2

In [14]:
import os
import time
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import pandas.api.types

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

import matplotlib.pyplot as plt

from tqdm import tqdm
from pqdm.threads import pqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from astropy.stats import sigma_clip
from scipy.signal import savgol_filter
from scipy.optimize import minimize
import scipy.stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT_PATH = "/kaggle/input/ariel-data-challenge-2025"
MODE = "test"

__t0 = time.perf_counter()

class Config:
    FEATURES = ['transit_depth', 'Rs', 'i']
    # FEATURES = ['transit_depth', 'Rs', 'i', 'P', 'scaled_depth']
    # FEATURES = ['transit_depth', 'Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"
    load_data = False  # Set to True to load from disk, False to generate
    DEBUG = False

    SCALE = 0.96
    SIGMA = 0.00055
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11 # 9
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

class ParticipantVisibleError(Exception):
    pass

def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    naive_mean: float,
    naive_sigma: float,
    fsg_sigma_true: float = 1e-6,
    airs_sigma_true: float = 1e-5,
    fgs_weight: float = 1,
) -> float:
    """
    This is a Gaussian Log Likelihood based metric. For a submission, which contains the predicted mean (x_hat) and variance (x_hat_std),
    we calculate the Gaussian Log-likelihood (GLL) value to the provided ground truth (x). We treat each pair of x_hat,
    x_hat_std as a 1D gaussian, meaning there will be 283 1D gaussian distributions, hence 283 values for each test spectrum,
    the GLL value for one spectrum is the sum of all of them.

    Inputs:
        - solution: Ground Truth spectra (from test set)
            - shape: (nsamples, n_wavelengths)
        - submission: Predicted spectra and errors (from participants)
            - shape: (nsamples, n_wavelengths*2)
        naive_mean: (float) mean from the train set.
        naive_sigma: (float) standard deviation from the train set.
        fsg_sigma_true: (float) standard deviation from the FSG1 instrument for the test set.
        airs_sigma_true: (float) standard deviation from the AIRS instrument for the test set.
        fgs_weight: (float) relative weight of the fgs channel
    """

    del solution[row_id_column_name]
    del submission[row_id_column_name]

    if submission.min().min() < 0:
        raise ParticipantVisibleError('Negative values in the submission')
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')

    n_wavelengths = len(solution.columns)
    if len(submission.columns) != n_wavelengths * 2:
        raise ParticipantVisibleError('Wrong number of columns in the submission')

    y_pred = submission.iloc[:, :n_wavelengths].values
    # Set a non-zero minimum sigma pred to prevent division by zero errors.
    sigma_pred = np.clip(submission.iloc[:, n_wavelengths:].values, a_min=10**-15, a_max=None)
    sigma_true = np.append(
        np.array(
            [
                fsg_sigma_true,
            ]
        ),
        np.ones(n_wavelengths - 1) * airs_sigma_true,
    )
    y_true = solution.values

    GLL_pred = scipy.stats.norm.logpdf(y_true, loc=y_pred, scale=sigma_pred)
    GLL_true = scipy.stats.norm.logpdf(y_true, loc=y_true, scale=sigma_true * np.ones_like(y_true))
    GLL_mean = scipy.stats.norm.logpdf(y_true, loc=naive_mean * np.ones_like(y_true), scale=naive_sigma * np.ones_like(y_true))

    # normalise the score, right now it becomes a matrix instead of a scalar.
    ind_scores = (GLL_pred - GLL_mean) / (GLL_true - GLL_mean)

    weights = np.append(np.array([fgs_weight]), np.ones(len(solution.columns) - 1))
    weights = weights * np.ones_like(ind_scores)
    submit_score = np.average(ind_scores, weights=weights)
    return float(np.clip(submit_score, 0.0, 1.0))

def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    if g1_max != 0: g1 /= g1_max
    if g2_max != 0: g2 /= g2_max
    phase1 = int(np.argmin(g1)); phase2 = int(np.argmax(g2)) + min_idx
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    """Возвращает вектор sigma_1 (для FGS1) длиной N_planets — мягкий множитель к cfg.SIGMA."""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    for single in preprocessed_data:
        # фазы по AIRS белой кривой — так же, как в модели
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        fgs = single[:, 0]
        oot = (fgs[: p1 - delta] if p1 - delta > 0 else np.empty(0, fgs.dtype))
        if p2 + delta < fgs.size:
            oot = np.concatenate([oot, fgs[p2 + delta :]])
        inn = fgs[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
        # относительная неопределённость глубины (в тех же ед., что s)
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    # мягкий множитель: корень, и узкий клип, чтобы не рисковать
    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.25)  # ±20–25% от базовой σ

    return k * cfg.SIGMA

def estimate_sigma_air(preprocessed_data, cfg):
    """Возвращает вектор sigma_air длиной N_planets — мягкий множитель к cfg.SIGMA для всех AIRS-каналов."""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        # белая кривая AIRS на бинированных данных (после всех твоих весов по λ)
        white = np.nanmean(single[:, 1:], axis=1)         # (n_bins,)
        white_s = savgol_filter(white, 20, 2)             # для фаз

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    # мягкий множитель вокруг медианы
    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.90, 1.20)  # ±10%–20%

    return k * cfg.SIGMA

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)

    def _apply_linear_corr(self, linear_corr, signal):

        coeffs = np.flip(linear_corr, axis=0)      # shape: (D, X, Y), D — старшая степень сначала
        x = signal.astype(np.float64, copy=False)  # считаем в float64 для стабильности
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]  # broadcast (X,Y) -> (T,X,Y)
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)  # in-place умножение
            out += coeffs[k]              # broadcast (X,Y)

        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        """
        Калибровка single-node сигнала.
        Политика масок: DEAD — маскируем, HOT — НЕ маскируем (оставляем в данных).
        """
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
    
        # --- load ---
        signal = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet"
        ).to_numpy()
        dark = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet"
        ).to_numpy()
        dead = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet"
        ).to_numpy()
        flat = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet"
        ).to_numpy()
        linear_corr = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet"
        ).values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])
    
        # --- reshape & ADC ---
        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset  # сохраняем твою формулу
    
        # HOT только для мониторинга, не для маскирования
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask
    
        # --- crop per sensor ---
        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]  # только для логов
    
        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]  # только для логов
    
        # --- non-neg clamp before linearity corr (как у тебя) ---
        np.maximum(signal, 0, out=signal)
    
        # --- linearity correction ---
        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))  # T, Y, λ
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)
    
        # --- dark subtraction с учётом паттерна интеграций ---
        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment
        signal[::2]  -= dark * even_scale
        signal[1::2] -= dark * odd_scale
    
        # --- APPLY FLAT (HOT-KEEP: не включаем hot в маску!) ---
        if sensor == "FGS1":
            flat_roi = flat.astype(signal.dtype, copy=False).copy()      # (12,12)
            bad = (dead) | ~np.isfinite(flat_roi) | (flat_roi == 0)      # ← ТОЛЬКО dead/invalid
            flat_roi[bad] = np.nan
            signal /= flat_roi
    
        elif sensor == "AIRS-CH0":
            y0, y1 = 10, 22
            flat_roi = flat[y0:y1, :].astype(signal.dtype, copy=False).copy()  # (12, λ)
            bad = (dead[y0:y1, :]) | ~np.isfinite(flat_roi) | (flat_roi == 0)  # ← ТОЛЬКО dead/invalid
            flat_roi[bad] = np.nan
            signal[:, y0:y1, :] /= flat_roi
    
        else:
            flat2 = flat.astype(signal.dtype, copy=False).copy()
            bad2 = (dead) | ~np.isfinite(flat2) | (flat2 == 0)                  # ← ТОЛЬКО dead/invalid
            flat2[bad2] = np.nan
            signal /= flat2
        # --- END FLAT ---
    
        # (опционально) логируем метрики hot/dead
        if getattr(self.cfg, "LOG_HOT_STATS", False):
            if not hasattr(self, "stats"):
                self.stats = []
            self.stats.append({
                "planet_id": int(planet_id),
                "sensor": sensor,
                "hot_frac": float(np.mean(hot)),
                "dead_frac": float(np.mean(dead)),
            })
    
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        mean_signal = np.nanmean(signal_roi, axis=1)

        cds_signal = mean_signal[1::2] - mean_signal[0::2]

        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)    # (n_bins, 1)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)   # (n_bins, 1)
            np.clip(binned, q_lo, q_hi, out=binned)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)                 # (λ, )
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var

            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]


        return binned

    def _process_planet_sensor(self, args):
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def process_all_data(self):
        args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
        preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
        preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        preprocessed_signal = np.concatenate(
            [np.stack(preprocessed_fgs1), np.stack(preprocessed_airs_ch0)], axis=2
        )
        return preprocessed_signal
    

class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index

        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        
        return error

    def predict(self, single_preprocessed_signal):
        signal_1d = single_preprocessed_signal[:, 1:].mean(axis=1)
        signal_1d = savgol_filter(signal_1d, 23, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        phase2 = min(len(signal_1d) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)    

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
        ]
        return np.array(predictions) * self.cfg.SCALE

StarInfo = pd.read_csv(ROOT_PATH + f"/{MODE}_star_info.csv")
StarInfo["planet_id"] = StarInfo["planet_id"].astype(int)
PlanetIds = StarInfo["planet_id"].tolist()
StarInfo = StarInfo.set_index("planet_id")
class SubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions1, predictions, sigma_fgs=None, sigma_air=None):
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2  # 283

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        submission_df.iloc[:, 0] = predictions
        submission_df.iloc[:, 1:283] = predictions1
        submission_df.to_csv("submission_1.csv")
        
        return submission_df

class SEBlock(nn.Module):
    def __init__(self, dim, reduction=16):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim // reduction, bias=False)
        self.fc2 = nn.Linear(dim // reduction, dim, bias=False)
        self.act = nn.SiLU()   # Or nn.ReLU()

    def forward(self, x):
        # Compute channel-wise attention
        w = x.mean(dim=0, keepdim=True)      # Global context (mean across batch)
        w = self.act(self.fc1(w))
        w = torch.sigmoid(self.fc2(w))
        return x * w   # Rescale input

class AttentionBlock(nn.Module):
    def __init__(self, dim, num_heads=4, dropout=0.1):
        super().__init__()
        self.attn = nn.MultiheadAttention(embed_dim=dim, num_heads=num_heads, batch_first=True)
        self.norm = nn.LayerNorm(dim)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        # Expect shape [batch, seq_len, dim], so expand if only [batch, dim]
        if x.dim() == 2:
            x = x.unsqueeze(1)   # -> [batch, 1, dim]
        attn_out, _ = self.attn(x, x, x)
        out = self.norm(x + self.dropout(attn_out))
        return out.squeeze(1)    # Back to [batch, dim]

class ResidualBlock2(nn.Module):
    def __init__(self, dim, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        identity = x
        out = self.relu(self.fc1(x))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.relu(out + identity)

class ResNetMLP2(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, output_dim=282, num_blocks=3, dropout_rate=0.2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[ResidualBlock2(hidden_dim, p=dropout_rate) for _ in range(num_blocks)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.output_layer(x)
        return x


def load_cv_models_and_scalers(directory):
    """
    Loads all cross-validation models and scalers from the specified directory.
    Args:
        directory (str): Path to the directory containing model and scaler files.
    Returns:
        all_models (list): List of loaded models.
        scaler_X: Loaded X scaler.
        scaler_y: Loaded y scaler.
    """
    import os
    import joblib
    # Load scalers
    scaler_X = joblib.load(os.path.join(directory, 'scaler_X.joblib'))
    scaler_y = joblib.load(os.path.join(directory, 'scaler_y.joblib'))

    # Load all CV models
    all_models = []
    model_params = {
        'input_dim': len(Config.FEATURES),  # Always use the current feature count
        'hidden_dim': 256,
        'output_dim': 282,
        'num_blocks': 35,
        'dropout_rate': 0.1
    }
    for fold in range(1, 11):
        model = ResNetMLP2(**model_params).double()
        model_path = os.path.join(directory, f'best_model_airs_cv_fold{fold}.pth')
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        model.eval()
        all_models.append(model)
    return all_models, scaler_X, scaler_y
    
all_models, scaler_X, scaler_y = load_cv_models_and_scalers('/kaggle/input/arial-dataset-new-version/ariel-2025-result/results_v37')

config = Config()
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

predictions_df = pd.DataFrame({
    "planet_id": PlanetIds,
    "transit_depth": predictions
})

input_df = pd.merge(predictions_df, StarInfo, on="planet_id", how="left")
input_df['scaled_depth'] = input_df['transit_depth'] / input_df['Rs']

X = input_df[Config.FEATURES].values.astype(np.float64)
X_scaled = scaler_X.transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float64)

# Generate average prediction from all CV models (on scaled X, then inverse transform)
with torch.no_grad():
    preds_scaled = [model(X_tensor).numpy() for model in all_models]
predictions1_scaled = np.mean(preds_scaled, axis=0)
predictions1 = scaler_y.inverse_transform(predictions1_scaled)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 66.09it/s]


In [15]:
class Config:
    FEATURES = ['transit_depth', 'Rs', 'i', 'P']

    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"
    load_data = False  # Set to True to load from disk, False to generate
    DEBUG = False

    SCALE = 0.96
    SIGMA = 0.00055
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11 # 9
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3


def load_cv_models_and_scalers(directory):
    """
    Loads all cross-validation models and scalers from the specified directory.
    Args:
        directory (str): Path to the directory containing model and scaler files.
    Returns:
        all_models (list): List of loaded models.
        scaler_X: Loaded X scaler.
        scaler_y: Loaded y scaler.
    """
    import os
    import joblib
    # Load scalers
    scaler_X = joblib.load(os.path.join(directory, 'scaler_X.joblib'))
    scaler_y = joblib.load(os.path.join(directory, 'scaler_y.joblib'))

    # Load all CV models
    all_models = []
    model_params = {
        'input_dim': len(Config.FEATURES),  # Always use the current feature count
        'hidden_dim': 256,
        'output_dim': 282,
        'num_blocks': 35,
        'dropout_rate': 0.1
    }
    for fold in range(1, 11):
        model = ResNetMLP2(**model_params).double()
        model_path = os.path.join(directory, f'best_model_airs_cv_fold{fold}.pth')
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        model.eval()
        all_models.append(model)
    return all_models, scaler_X, scaler_y
    
all_models, scaler_X, scaler_y = load_cv_models_and_scalers('/kaggle/input/arial-dataset-new-version/ariel-2025-result/results_sv44')
config = Config()
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

predictions_df = pd.DataFrame({
    "planet_id": PlanetIds,
    "transit_depth": predictions
})

input_df = pd.merge(predictions_df, StarInfo, on="planet_id", how="left")
X = input_df[Config.FEATURES].values.astype(np.float64)
X_scaled = scaler_X.transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float64)

# Generate average prediction from all CV models (on scaled X, then inverse transform)
with torch.no_grad():
    preds_scaled = [model(X_tensor).numpy() for model in all_models]

# Correctly average the predictions
predictions2_scaled = np.mean(preds_scaled, axis=0)

# Correctly inverse transform the *averaged* predictions
predictions2 = scaler_y.inverse_transform(predictions2_scaled)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 64.16it/s]


In [16]:
# Combine the predictions into a single array
all_predictions = np.array([predictions1, predictions2])

# Calculate the mean across the models (axis=0)
final_predictions = np.mean(all_predictions, axis=0)

submission_generator = SubmissionGenerator(config)
submission = submission_generator.create(final_predictions, predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)


In [17]:
xgb = pd.read_csv("submission_1.csv") # 370
nn =  pd.read_csv("submission_0.csv") # 374

In [18]:


# Identify columns
wl_cols    = [c for c in xgb.columns if c.startswith("wl_")]
sigma_cols = [c for c in xgb.columns if c.startswith("sigma_")]

# Weighted blends
wl_blend    = 0.2 * xgb[wl_cols] + 0.8 * nn[wl_cols]
sigma_blend = 0.4 * xgb[sigma_cols] + 0.6 * nn[sigma_cols]

# Combine with planet_id
submission = pd.concat(
    [xgb[["planet_id"]], wl_blend, sigma_blend],
    axis=1
)

# Save final submission
submission.to_csv("submission_380.csv", index=False)
print("Saved submission.csv with shape:", submission.shape)



Saved submission.csv with shape: (1, 567)


In [19]:
submission

,planet_id,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
0,1103775,0.015862,0.015643,0.01578,0.015691,0.015677,0.015628,0.015744,0.015734,0.015844,...,0.000569,0.000569,0.000569,0.000569,0.000569,0.000569,0.000569,0.000569,0.000569,0.000569


# series 2

In [20]:
import os
import time
import itertools
import multiprocessing as mp

import numpy as np
import pandas as pd
import pandas.api.types

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader, TensorDataset, random_split

import matplotlib.pyplot as plt

from tqdm import tqdm
from pqdm.threads import pqdm

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import mean_squared_error

from astropy.stats import sigma_clip
from scipy.signal import savgol_filter
from scipy.optimize import minimize
import scipy.stats
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
from sklearn.preprocessing import StandardScaler
import pandas as pd
import numpy as np
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

ROOT_PATH = "/kaggle/input/ariel-data-challenge-2025"
MODE = "test"

__t0 = time.perf_counter()

class Config:
    FEATURES = ['transit_depth', 'Rs', 'i']
    # FEATURES = ['transit_depth', 'Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"
    load_data = True  # Set to True to load from disk, False to generate
    DEBUG = False

    SCALE = 0.96
    SIGMA = 0.00055
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11 # 9
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

class ParticipantVisibleError(Exception):
    pass

def score(
    solution: pd.DataFrame,
    submission: pd.DataFrame,
    row_id_column_name: str,
    naive_mean: float,
    naive_sigma: float,
    fsg_sigma_true: float = 1e-6,
    airs_sigma_true: float = 1e-5,
    fgs_weight: float = 1,
) -> float:
    """
    This is a Gaussian Log Likelihood based metric. For a submission, which contains the predicted mean (x_hat) and variance (x_hat_std),
    we calculate the Gaussian Log-likelihood (GLL) value to the provided ground truth (x). We treat each pair of x_hat,
    x_hat_std as a 1D gaussian, meaning there will be 283 1D gaussian distributions, hence 283 values for each test spectrum,
    the GLL value for one spectrum is the sum of all of them.

    Inputs:
        - solution: Ground Truth spectra (from test set)
            - shape: (nsamples, n_wavelengths)
        - submission: Predicted spectra and errors (from participants)
            - shape: (nsamples, n_wavelengths*2)
        naive_mean: (float) mean from the train set.
        naive_sigma: (float) standard deviation from the train set.
        fsg_sigma_true: (float) standard deviation from the FSG1 instrument for the test set.
        airs_sigma_true: (float) standard deviation from the AIRS instrument for the test set.
        fgs_weight: (float) relative weight of the fgs channel
    """

    del solution[row_id_column_name]
    del submission[row_id_column_name]

    if submission.min().min() < 0:
        raise ParticipantVisibleError('Negative values in the submission')
    for col in submission.columns:
        if not pandas.api.types.is_numeric_dtype(submission[col]):
            raise ParticipantVisibleError(f'Submission column {col} must be a number')

    n_wavelengths = len(solution.columns)
    if len(submission.columns) != n_wavelengths * 2:
        raise ParticipantVisibleError('Wrong number of columns in the submission')

    y_pred = submission.iloc[:, :n_wavelengths].values
    # Set a non-zero minimum sigma pred to prevent division by zero errors.
    sigma_pred = np.clip(submission.iloc[:, n_wavelengths:].values, a_min=10**-15, a_max=None)
    sigma_true = np.append(
        np.array(
            [
                fsg_sigma_true,
            ]
        ),
        np.ones(n_wavelengths - 1) * airs_sigma_true,
    )
    y_true = solution.values

    GLL_pred = scipy.stats.norm.logpdf(y_true, loc=y_pred, scale=sigma_pred)
    GLL_true = scipy.stats.norm.logpdf(y_true, loc=y_true, scale=sigma_true * np.ones_like(y_true))
    GLL_mean = scipy.stats.norm.logpdf(y_true, loc=naive_mean * np.ones_like(y_true), scale=naive_sigma * np.ones_like(y_true))

    # normalise the score, right now it becomes a matrix instead of a scalar.
    ind_scores = (GLL_pred - GLL_mean) / (GLL_true - GLL_mean)

    weights = np.append(np.array([fgs_weight]), np.ones(len(solution.columns) - 1))
    weights = weights * np.ones_like(ind_scores)
    submit_score = np.average(ind_scores, weights=weights)
    return float(np.clip(submit_score, 0.0, 1.0))

def _phase_detector_signal(signal, cfg):
    sl = cfg.MODEL_PHASE_DETECTION_SLICE
    min_idx = int(np.argmin(signal[sl])) + sl.start
    s1 = signal[:min_idx]; s2 = signal[min_idx:]
    if s1.size < 3 or s2.size < 3:
        return 0, len(signal) - 1
    g1 = np.gradient(s1); g1_max = np.max(g1) if np.size(g1) else 0.0
    g2 = np.gradient(s2); g2_max = np.max(g2) if np.size(g2) else 0.0
    if g1_max != 0: g1 /= g1_max
    if g2_max != 0: g2 /= g2_max
    phase1 = int(np.argmin(g1)); phase2 = int(np.argmax(g2)) + min_idx
    return phase1, phase2

def estimate_sigma_fgs(preprocessed_data, cfg):
    """Возвращает вектор sigma_1 (для FGS1) длиной N_planets — мягкий множитель к cfg.SIGMA."""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12
    for single in preprocessed_data:
        # фазы по AIRS белой кривой — так же, как в модели
        air_white = savgol_filter(single[:, 1:].mean(axis=1), 20, 2)
        p1, p2 = _phase_detector_signal(air_white, cfg)
        p1 = max(delta, p1)
        p2 = min(len(air_white) - delta - 1, p2)

        fgs = single[:, 0]
        oot = (fgs[: p1 - delta] if p1 - delta > 0 else np.empty(0, fgs.dtype))
        if p2 + delta < fgs.size:
            oot = np.concatenate([oot, fgs[p2 + delta :]])
        inn = fgs[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(fgs))
        # относительная неопределённость глубины (в тех же ед., что s)
        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    # мягкий множитель: корень, и узкий клип, чтобы не рисковать
    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.8, 1.25)  # ±20–25% от базовой σ

    return k * cfg.SIGMA

def estimate_sigma_air(preprocessed_data, cfg):
    """Возвращает вектор sigma_air длиной N_planets — мягкий множитель к cfg.SIGMA для всех AIRS-каналов."""
    sig_rel = []
    delta = cfg.MODEL_OPTIMIZATION_DELTA
    eps = 1e-12

    for single in preprocessed_data:
        # белая кривая AIRS на бинированных данных (после всех твоих весов по λ)
        white = np.nanmean(single[:, 1:], axis=1)         # (n_bins,)
        white_s = savgol_filter(white, 20, 2)             # для фаз

        p1, p2 = _phase_detector_signal(white_s, cfg)
        p1 = max(delta, p1)
        p2 = min(len(white) - delta - 1, p2)

        oot_left = white[: p1 - delta] if p1 - delta > 0 else np.empty(0, white.dtype)
        oot_right = white[p2 + delta :] if (p2 + delta) < white.size else np.empty(0, white.dtype)
        oot = np.concatenate([oot_left, oot_right]) if (oot_left.size + oot_right.size) else oot_left
        inn = white[p1 + delta : max(p1 + delta, p2 - delta)]

        if oot.size == 0 or inn.size == 0:
            sig_rel.append(np.nan); continue

        n_oot, n_in = len(oot), len(inn)
        var_oot = np.nanvar(oot, ddof=1)
        var_in  = np.nanvar(inn, ddof=1)
        oot_mean = float(np.nanmean(oot)) if np.isfinite(np.nanmean(oot)) else float(np.nanmean(white))

        sigma_rel = np.sqrt(var_oot / max(n_oot,1) + var_in / max(n_in,1)) / max(oot_mean, eps)
        sig_rel.append(sigma_rel)

    s = np.asarray(sig_rel, dtype=float)
    mask = np.isfinite(s) & (s > 0)
    med = float(np.nanmedian(s[mask])) if mask.any() else 1.0

    # мягкий множитель вокруг медианы
    k = np.ones_like(s)
    if med > 0 and np.isfinite(med):
        k[mask] = np.sqrt(s[mask] / med)
    k = np.clip(k, 0.90, 1.20)  # ±10%–20%

    return k * cfg.SIGMA

class SignalProcessor:
    def __init__(self, config):
        self.cfg = config
        self.adc_info = pd.read_csv(f"{self.cfg.DATA_PATH}/adc_info.csv")
        self.planet_ids = pd.read_csv(f'{self.cfg.DATA_PATH}/{self.cfg.DATASET}_star_info.csv', index_col='planet_id').index.astype(int)

    def _apply_linear_corr(self, linear_corr, signal):

        coeffs = np.flip(linear_corr, axis=0)      # shape: (D, X, Y), D — старшая степень сначала
        x = signal.astype(np.float64, copy=False)  # считаем в float64 для стабильности
        out = np.empty_like(x, dtype=np.float64)
        out[...] = coeffs[0]  # broadcast (X,Y) -> (T,X,Y)
        for k in range(1, coeffs.shape[0]):
            np.multiply(out, x, out=out)  # in-place умножение
            out += coeffs[k]              # broadcast (X,Y)

        return out.astype(signal.dtype, copy=False)

    def _calibrate_single_signal(self, planet_id, sensor):
        """
        Калибровка single-node сигнала.
        Политика масок: DEAD — маскируем, HOT — НЕ маскируем (оставляем в данных).
        """
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
    
        # --- load ---
        signal = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_signal_0.parquet"
        ).to_numpy()
        dark = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dark.parquet"
        ).to_numpy()
        dead = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/dead.parquet"
        ).to_numpy()
        flat = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/flat.parquet"
        ).to_numpy()
        linear_corr = pd.read_parquet(
            f"{self.cfg.DATA_PATH}/{self.cfg.DATASET}/{planet_id}/{sensor}_calibration_0/linear_corr.parquet"
        ).values.astype(np.float64).reshape(sensor_cfg["linear_corr_shape"])
    
        # --- reshape & ADC ---
        signal = signal.reshape(sensor_cfg["raw_shape"])
        gain = self.adc_info[f"{sensor}_adc_gain"].iloc[0]
        offset = self.adc_info[f"{sensor}_adc_offset"].iloc[0]
        signal = signal / gain + offset  # сохраняем твою формулу
    
        # HOT только для мониторинга, не для маскирования
        hot = sigma_clip(dark, sigma=5, maxiters=5).mask
    
        # --- crop per sensor ---
        if sensor == "AIRS-CH0":
            signal = signal[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            linear_corr = linear_corr[:, :, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dark = dark[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            dead = dead[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            flat = flat[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]
            hot = hot[:, self.cfg.CUT_INF : self.cfg.CUT_SUP]  # только для логов
    
        if sensor == "FGS1":
            y0, y1, x0, x1 = 10, 22, 10, 22
            signal = signal[:, y0:y1, x0:x1]
            dark   = dark[y0:y1, x0:x1]
            dead   = dead[y0:y1, x0:x1]
            flat   = flat[y0:y1, x0:x1]
            linear_corr = linear_corr[:, y0:y1, x0:x1]
            hot    = hot[y0:y1, x0:x1]  # только для логов
    
        # --- non-neg clamp before linearity corr (как у тебя) ---
        np.maximum(signal, 0, out=signal)
    
        # --- linearity correction ---
        if sensor == "FGS1":
            signal = self._apply_linear_corr(linear_corr, signal)
        elif sensor == "AIRS-CH0":
            sl = (slice(None), slice(10, 22), slice(None))  # T, Y, λ
            signal[sl] = self._apply_linear_corr(linear_corr[:, 10:22, :], signal[sl])
        else:
            signal = self._apply_linear_corr(linear_corr, signal)
    
        # --- dark subtraction с учётом паттерна интеграций ---
        base_dt, increment = sensor_cfg["dt_pattern"]
        even_scale = base_dt
        odd_scale  = base_dt + increment
        signal[::2]  -= dark * even_scale
        signal[1::2] -= dark * odd_scale
    
        # --- APPLY FLAT (HOT-KEEP: не включаем hot в маску!) ---
        if sensor == "FGS1":
            flat_roi = flat.astype(signal.dtype, copy=False).copy()      # (12,12)
            bad = (dead) | ~np.isfinite(flat_roi) | (flat_roi == 0)      # ← ТОЛЬКО dead/invalid
            flat_roi[bad] = np.nan
            signal /= flat_roi
    
        elif sensor == "AIRS-CH0":
            y0, y1 = 10, 22
            flat_roi = flat[y0:y1, :].astype(signal.dtype, copy=False).copy()  # (12, λ)
            bad = (dead[y0:y1, :]) | ~np.isfinite(flat_roi) | (flat_roi == 0)  # ← ТОЛЬКО dead/invalid
            flat_roi[bad] = np.nan
            signal[:, y0:y1, :] /= flat_roi
    
        else:
            flat2 = flat.astype(signal.dtype, copy=False).copy()
            bad2 = (dead) | ~np.isfinite(flat2) | (flat2 == 0)                  # ← ТОЛЬКО dead/invalid
            flat2[bad2] = np.nan
            signal /= flat2
        # --- END FLAT ---
    
        # (опционально) логируем метрики hot/dead
        if getattr(self.cfg, "LOG_HOT_STATS", False):
            if not hasattr(self, "stats"):
                self.stats = []
            self.stats.append({
                "planet_id": int(planet_id),
                "sensor": sensor,
                "hot_frac": float(np.mean(hot)),
                "dead_frac": float(np.mean(dead)),
            })
    
        return signal

    def _preprocess_calibrated_signal(self, calibrated_signal, sensor):
        sensor_cfg = self.cfg.SENSOR_CONFIG[sensor]
        binning = sensor_cfg["binning"]

        if sensor == "AIRS-CH0":
            signal_roi = calibrated_signal[:, 10:22, :]
        elif sensor == "FGS1":
            signal_roi = calibrated_signal[:, 10:22, 10:22]
            signal_roi = signal_roi.reshape(signal_roi.shape[0], -1)
        
        mean_signal = np.nanmean(signal_roi, axis=1)

        cds_signal = mean_signal[1::2] - mean_signal[0::2]

        n_bins = cds_signal.shape[0] // binning
        binned = np.array([
            cds_signal[j*binning : (j+1)*binning].mean(axis=0) 
            for j in range(n_bins)
        ])

        if sensor == "AIRS-CH0":
            q_lo = np.nanpercentile(binned, 5.0, axis=1, keepdims=True)    # (n_bins, 1)
            q_hi = np.nanpercentile(binned, 95.0, axis=1, keepdims=True)   # (n_bins, 1)
            np.clip(binned, q_lo, q_hi, out=binned)

        if sensor == "FGS1":
            binned = binned.reshape((binned.shape[0], 1))

        if sensor == "AIRS-CH0":
            var = np.nanvar(binned, axis=0, ddof=1)                 # (λ, )
            med = np.nanmedian(var)
            safe_var = np.where(~np.isfinite(var) | (var <= 0), med if (np.isfinite(med) and med > 0) else 1.0, var)
            w = 1.0 / safe_var

            lo, hi = np.nanpercentile(w, 5.0), np.nanpercentile(w, 95.0)
            if np.isfinite(lo) and np.isfinite(hi) and lo < hi:
                w = np.clip(w, lo, hi)

            M = binned.shape[1]
            s = np.nansum(w)
            if np.isfinite(s) and s > 0:
                w = w * (M / s)
            else:
                w = np.ones_like(w)

            binned *= w[None, :]


        return binned

    def _process_planet_sensor(self, args):
        planet_id, sensor = args['planet_id'], args['sensor']
        calibrated = self._calibrate_single_signal(planet_id, sensor)
        preprocessed = self._preprocess_calibrated_signal(calibrated, sensor)
        return preprocessed

    def process_all_data(self):
        args_fgs1 = [dict(planet_id=planet_id, sensor="FGS1") for planet_id in self.planet_ids]
        preprocessed_fgs1 = pqdm(args_fgs1, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        args_airs_ch0 = [dict(planet_id=planet_id, sensor="AIRS-CH0") for planet_id in self.planet_ids]
        preprocessed_airs_ch0 = pqdm(args_airs_ch0, self._process_planet_sensor, n_jobs=self.cfg.N_JOBS)

        preprocessed_signal = np.concatenate(
            [np.stack(preprocessed_fgs1), np.stack(preprocessed_airs_ch0)], axis=2
        )
        return preprocessed_signal
    

class TransitModel:
    def __init__(self, config):
        self.cfg = config

    def _phase_detector(self, signal):
        search_slice = self.cfg.MODEL_PHASE_DETECTION_SLICE
        min_index = np.argmin(signal[search_slice]) + search_slice.start
        
        signal1 = signal[:min_index]
        signal2 = signal[min_index:]

        grad1 = np.gradient(signal1)
        grad1 /= grad1.max()
        
        grad2 = np.gradient(signal2)
        grad2 /= grad2.max()

        phase1 = np.argmin(grad1)
        phase2 = np.argmax(grad2) + min_index

        return phase1, phase2
    
    def _objective_function(self, s, signal, phase1, phase2):
        delta = self.cfg.MODEL_OPTIMIZATION_DELTA
        power = self.cfg.MODEL_POLYNOMIAL_DEGREE

        if phase1 - delta <= 0 or phase2 + delta >= len(signal) or phase2 - delta - (phase1 + delta) < 5:
            delta = 2

        y = np.concatenate([
            signal[: phase1 - delta],
            signal[phase1 + delta : phase2 - delta] * (1 + s),
            signal[phase2 + delta :]
        ])
        x = np.arange(len(y))

        coeffs = np.polyfit(x, y, deg=power)
        poly = np.poly1d(coeffs)
        error = np.abs(poly(x) - y).mean()
        
        return error

    def predict(self, single_preprocessed_signal):
        signal_1d = single_preprocessed_signal[:, 1:].mean(axis=1)
        signal_1d = savgol_filter(signal_1d, 23, 2)
        
        phase1, phase2 = self._phase_detector(signal_1d)

        phase1 = max(self.cfg.MODEL_OPTIMIZATION_DELTA, phase1)
        phase2 = min(len(signal_1d) - self.cfg.MODEL_OPTIMIZATION_DELTA - 1, phase2)    

        result = minimize(
            fun=self._objective_function,
            x0=[0.0001],
            args=(signal_1d, phase1, phase2),
            method="Nelder-Mead"
        )
        
        return result.x[0]

    def predict_all(self, preprocessed_signals):
        predictions = [
            self.predict(preprocessed_signal)
            for preprocessed_signal in tqdm(preprocessed_signals)
        ]
        return np.array(predictions) * self.cfg.SCALE

StarInfo = pd.read_csv(ROOT_PATH + f"/{MODE}_star_info.csv")
StarInfo["planet_id"] = StarInfo["planet_id"].astype(int)
PlanetIds = StarInfo["planet_id"].tolist()
StarInfo = StarInfo.set_index("planet_id")
class SubmissionGenerator:
    def __init__(self, config):
        self.cfg = config
        self.sample_submission = pd.read_csv("/kaggle/input/ariel-data-challenge-2025/sample_submission.csv", index_col="planet_id")

    def create(self, predictions1, predictions, sigma_fgs=None, sigma_air=None):
        planet_ids = self.sample_submission.index
        n_mu = self.sample_submission.shape[1] // 2  # 283

        preds = np.asarray(predictions, dtype=float).reshape(-1)
        mu = np.tile(preds.reshape(-1, 1), (1, n_mu))
        mu = np.clip(mu, 0, None)

        sigmas = np.full_like(mu, self.cfg.SIGMA, dtype=float)
        if sigma_fgs is not None:
            sigma_fgs = np.asarray(sigma_fgs, dtype=float).reshape(-1)
            sigmas[:, 0] = np.clip(sigma_fgs, 1e-6, 0.1)
        if sigma_air is not None:
            sigma_air = np.asarray(sigma_air, dtype=float).reshape(-1, 1)
            sigmas[:, 1:] = np.clip(sigma_air, 1e-6, 0.1)

        submission_df = pd.DataFrame(
            np.concatenate([mu, sigmas], axis=1),
            columns=self.sample_submission.columns,
            index=planet_ids
        )
        submission_df.iloc[:, 0] = predictions
        submission_df.iloc[:, 1:283] = predictions1
        submission_df.to_csv("submission.csv")
        
        return submission_df

class ResidualBlock2(nn.Module):
    def __init__(self, dim, p=0.2):
        super().__init__()
        self.fc1 = nn.Linear(dim, dim)
        self.fc2 = nn.Linear(dim, dim)
        self.relu = nn.ReLU()
        self.dropout = nn.Dropout(p)

    def forward(self, x):
        identity = x
        out = self.relu(self.fc1(x))
        out = self.dropout(out)
        out = self.fc2(out)
        return self.relu(out + identity)

class ResNetMLP2(nn.Module):
    def __init__(self, input_dim=3, hidden_dim=128, output_dim=282, num_blocks=3, dropout_rate=0.2):
        super().__init__()
        self.input_layer = nn.Linear(input_dim, hidden_dim)
        self.blocks = nn.Sequential(*[ResidualBlock2(hidden_dim, p=dropout_rate) for _ in range(num_blocks)])
        self.output_layer = nn.Linear(hidden_dim, output_dim)

    def forward(self, x):
        x = self.input_layer(x)
        x = self.blocks(x)
        x = self.output_layer(x)
        return x

# Execute the training
# Train and get scalers
# all_models, scaler_X, scaler_y = train_new_model()

def load_cv_models_and_scalers(directory):
    """
    Loads all cross-validation models and scalers from the specified directory.
    Args:
        directory (str): Path to the directory containing model and scaler files.
    Returns:
        all_models (list): List of loaded models.
        scaler_X: Loaded X scaler.
        scaler_y: Loaded y scaler.
    """
    import os
    import joblib
    # Load scalers
    scaler_X = joblib.load(os.path.join(directory, 'scaler_X.joblib'))
    scaler_y = joblib.load(os.path.join(directory, 'scaler_y.joblib'))

    # Load all CV models
    all_models = []
    model_params = {
        'input_dim': len(Config.FEATURES),  # Always use the current feature count
        'hidden_dim': 256,
        'output_dim': 282,
        'num_blocks': 80,
        'dropout_rate': 0.3
    }
    for fold in range(1, 11):
        model = ResNetMLP2(**model_params).double()
        model_path = os.path.join(directory, f'best_model_airs_cv_fold{fold}.pth')
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        model.eval()
        all_models.append(model)
    return all_models, scaler_X, scaler_y
    
all_models, scaler_X, scaler_y = load_cv_models_and_scalers('/kaggle/input/ariel-2025-result/results_3_v25')
config = Config()
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

predictions_df = pd.DataFrame({
    "planet_id": PlanetIds,
    "transit_depth": predictions
})

input_df = pd.merge(predictions_df, StarInfo, on="planet_id", how="left")
X = input_df[Config.FEATURES].values.astype(np.float64)
X_scaled = scaler_X.transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float64)

# Generate average prediction from all CV models (on scaled X, then inverse transform)
with torch.no_grad():
    preds_scaled = [model(X_tensor).numpy() for model in all_models]
predictions1_scaled = np.mean(preds_scaled, axis=0)
predictions1 = scaler_y.inverse_transform(predictions1_scaled)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 77.76it/s]


In [21]:
class Config:
    # FEATURES = ['transit_depth', 'Rs', 'i']
    FEATURES = ['transit_depth', 'Rs', 'Ms', 'Ts', 'Mp', 'e', 'P', 'sma', 'i']
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025'
    DATASET = "test"
    load_data = True  # Set to True to load from disk, False to generate
    DEBUG = False

    SCALE = 0.96
    SIGMA = 0.00055
    
    CUT_INF = 39
    CUT_SUP = 321
    
    SENSOR_CONFIG = {
        "AIRS-CH0": {
            "raw_shape": [11250, 32, 356],
            "calibrated_shape": [1, 32, CUT_SUP - CUT_INF],
            "linear_corr_shape": (6, 32, 356),
            "dt_pattern": (0.1, 4.5), 
            "binning": 30
        },
        "FGS1": {
            "raw_shape": [135000, 32, 32],
            "calibrated_shape": [1, 32, 32],
            "linear_corr_shape": (6, 32, 32),
            "dt_pattern": (0.1, 0.1),
            "binning": 30 * 12
        }
    }
    
    MODEL_PHASE_DETECTION_SLICE = slice(30, 140)
    MODEL_OPTIMIZATION_DELTA = 11 # 9
    MODEL_POLYNOMIAL_DEGREE = 3
    
    N_JOBS = 3

def load_cv_models_and_scalers(directory):
    """
    Loads all cross-validation models and scalers from the specified directory.
    Args:
        directory (str): Path to the directory containing model and scaler files.
    Returns:
        all_models (list): List of loaded models.
        scaler_X: Loaded X scaler.
        scaler_y: Loaded y scaler.
    """
    import os
    import joblib
    # Load scalers
    scaler_X = joblib.load(os.path.join(directory, 'scaler_X.joblib'))
    scaler_y = joblib.load(os.path.join(directory, 'scaler_y.joblib'))

    # Load all CV models
    all_models = []
    model_params = {
        'input_dim': len(Config.FEATURES),  # Always use the current feature count
        'hidden_dim': 256,
        'output_dim': 282,
        'num_blocks': 80,
        'dropout_rate': 0.3
    }
    for fold in range(1, 11):
        model = ResNetMLP2(**model_params).double()
        model_path = os.path.join(directory, f'best_model_airs_cv_fold{fold}.pth')
        model.load_state_dict(torch.load(model_path, map_location=torch.device('cpu')))
        model.eval()
        all_models.append(model)
    return all_models, scaler_X, scaler_y
    
all_models, scaler_X, scaler_y = load_cv_models_and_scalers('/kaggle/input/ariel-2025-result/results_9_v26')
config = Config()
signal_processor = SignalProcessor(config)
preprocessed_data = signal_processor.process_all_data()

model = TransitModel(config)
predictions = model.predict_all(preprocessed_data)
sigma_fgs_vec = estimate_sigma_fgs(preprocessed_data, config)
sigma_air_vec = estimate_sigma_air(preprocessed_data, config)

predictions_df = pd.DataFrame({
    "planet_id": PlanetIds,
    "transit_depth": predictions
})

input_df = pd.merge(predictions_df, StarInfo, on="planet_id", how="left")
X = input_df[Config.FEATURES].values.astype(np.float64)
X_scaled = scaler_X.transform(X)
X_tensor = torch.tensor(X_scaled, dtype=torch.float64)

# Generate average prediction from all CV models (on scaled X, then inverse transform)
with torch.no_grad():
    preds_scaled = [model(X_tensor).numpy() for model in all_models]

# Correctly average the predictions
predictions2_scaled = np.mean(preds_scaled, axis=0)

# Correctly inverse transform the *averaged* predictions
predictions2 = scaler_y.inverse_transform(predictions2_scaled)

all_predictions = np.array([predictions1, predictions2])

# Calculate the mean across the models (axis=0)
final_predictions = np.mean(all_predictions, axis=0)

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

QUEUEING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

PROCESSING TASKS | :   0%|          | 0/1 [00:00<?, ?it/s]

COLLECTING RESULTS | :   0%|          | 0/1 [00:00<?, ?it/s]

100%|██████████| 1/1 [00:00<00:00, 63.19it/s]


In [22]:
submission_generator = SubmissionGenerator(config)
submission = submission_generator.create(final_predictions, predictions, sigma_fgs=sigma_fgs_vec, sigma_air=sigma_air_vec)

submission.to_csv("submission_nn.csv")
__t1 = time.perf_counter()
elapsed = __t1 - __t0
print(f"[TIMING] total runtime: {elapsed:.2f} s ({elapsed/60:.2f} min)")
pd.read_csv("submission.csv")

[TIMING] total runtime: 29.63 s (0.49 min)


,planet_id,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
0,1103775,0.016123,0.015649,0.015633,0.015638,0.015687,0.015667,0.015721,0.01566,0.015653,...,0.00055,0.00055,0.00055,0.00055,0.00055,0.00055,0.00055,0.00055,0.00055,0.00055


In [23]:
import os
import pickle
import numpy as np
import pandas as pd
import polars as pl

import scipy.stats
import xgboost as xgb
from sklearn.model_selection import StratifiedKFold
from sklearn.multioutput import MultiOutputRegressor
from sklearn.preprocessing import PolynomialFeatures
from tqdm import tqdm
from joblib import Parallel, delayed
import multiprocessing

# --- Set to True for training, False for submission ---
TRAIN = False 
# Set to True to use a small subset for faster debugging
DEBUG = False 

class Config:
    DATA_PATH = '/kaggle/input/ariel-data-challenge-2025/'
    OUTPUT_PATH = '/kaggle/input/5-fold-arial-xgb/ariel_kfold_models' # Save models to /kaggle/working/
    TRAIN_LABELS_PATH = os.path.join(DATA_PATH, 'train.csv')
    TRAIN_STAR_INFO_PATH = os.path.join(DATA_PATH, 'train_star_info.csv')
    TEST_STAR_INFO_PATH = os.path.join(DATA_PATH, 'test_star_info.csv')
    SAMPLE_SUBMISSION_PATH = os.path.join(DATA_PATH, 'sample_submission.csv')
    N_FOLDS = 5
    RANDOM_STATE = 42
    
    XGB_PARAMS_MEAN = {
        'objective': 'reg:squarederror',
        'n_estimators': 1200, # Slightly more estimators for smaller fold data
        'learning_rate': 0.02,
        'max_depth': 7,
        'subsample': 0.8,
        'colsample_bytree': 0.7,
        'random_state': RANDOM_STATE,
        'tree_method': 'hist',
        'n_jobs': -1, 
    }
    
    XGB_PARAMS_SIGMA = {
        'objective': 'reg:squarederror',
        'n_estimators': 600,
        'learning_rate': 0.025,
        'max_depth': 6,
        'subsample': 0.7,
        'colsample_bytree': 0.6,
        'random_state': RANDOM_STATE,
        'tree_method': 'hist',
        'n_jobs': -1,
    }
    
    CALIBRATION_SCALING_FACTORS = [0.8, 0.9, 1.0, 1.1, 1.2]
    CALIBRATION_ADDITIVE_FACTORS = [0.0001, 0.0002, 0.0003, 0.0004, 0.0005]

config = Config()

# --- All helper functions (extract_light_curve_with_aperture, process_single_planet, etc.) remain the same ---
# (I've omitted them here for brevity, but they should be included in your script exactly as they were)
def extract_light_curve_with_aperture(signal_df: pl.DataFrame, instrument: str):
    if instrument == 'FGS1':
        IMAGE_SHAPE = (32, 32)
        APERTURE_RADIUS = 3 
    else: # AIRS-CH0
        IMAGE_SHAPE = (32, 356)
        APERTURE_HEIGHT = 4 

    try:
        images = signal_df.to_numpy(zero_copy_only=True).reshape(-1, *IMAGE_SHAPE)
    except Exception:
        images = signal_df.to_numpy().reshape(-1, *IMAGE_SHAPE)
        
    median_frame = np.median(images, axis=0)

    if instrument == 'FGS1':
        center_y, center_x = np.unravel_index(np.argmax(median_frame), median_frame.shape)
        y_start = max(0, center_y - APERTURE_RADIUS)
        y_end = min(IMAGE_SHAPE[0], center_y + APERTURE_RADIUS + 1)
        x_start = max(0, center_x - APERTURE_RADIUS)
        x_end = min(IMAGE_SHAPE[1], center_x + APERTURE_RADIUS + 1)
        aperture_flux = images[:, y_start:y_end, x_start:x_end].sum(axis=(1, 2))
    else: # AIRS-CH0
        vertical_profile = median_frame.sum(axis=1)
        center_y = np.argmax(vertical_profile)
        y_start = max(0, center_y - APERTURE_HEIGHT)
        y_end = min(IMAGE_SHAPE[0], center_y + APERTURE_HEIGHT + 1)
        aperture_flux = images[:, y_start:y_end, :].sum(axis=(1, 2))
        
    return aperture_flux.astype(np.float32)

def process_single_planet(planet_id, dataset, instrument):
    planet_signals = []
    for obs_count in range(2):
        path = f'/kaggle/input/ariel-data-challenge-2025/{dataset}/{int(planet_id)}/{instrument}_signal_{obs_count}.parquet'
        if os.path.exists(path):
            signal_df = pl.read_parquet(path)
            aperture_flux = extract_light_curve_with_aperture(signal_df, instrument)
            net_signal = aperture_flux[1::2] - aperture_flux[0::2]
            planet_signals.append(net_signal)
    return planet_id, planet_signals

def load_all_observations(dataset, planet_ids, instrument='FGS1'):
    print(f"Loading ALL observations for {instrument} in {dataset} set...")
    results = Parallel(n_jobs=-1)(
        delayed(process_single_planet)(pid, dataset, instrument) 
        for pid in tqdm(planet_ids, desc=f"Dispatching {instrument} jobs")
    )
    return {planet_id: signals for planet_id, signals in results if signals}

def autocorrelation(x, lag=1):
    return np.corrcoef(x[:-lag], x[lag:])[0, 1]

def get_detrended_features(raw_data, transit_slice):
    time_axis = np.arange(raw_data.shape[0])
    out_of_transit_mask = np.ones_like(time_axis, dtype=bool)
    out_of_transit_mask[transit_slice] = False
    coeffs = np.polyfit(time_axis[out_of_transit_mask], raw_data[out_of_transit_mask], 2)
    poly_fit = np.poly1d(coeffs)
    trend = poly_fit(time_axis).clip(1e-6)
    normalized_data = raw_data / trend
    detrended_transit = normalized_data[transit_slice]
    return {
        'detrended_std': detrended_transit.std(), 
        'detrended_skew': scipy.stats.skew(detrended_transit),
        'detrended_kurtosis': scipy.stats.kurtosis(detrended_transit)
    }

def add_physics_and_interaction_features(features_df):
    print("Adding physics-informed and interaction features...")
    df = features_df.copy()
    transit_depth_proxy = df['fgs_depth_mean'].clip(0)
    df['planet_star_radius_ratio'] = np.sqrt(transit_depth_proxy)
    b = df['impact_parameter_b']
    p = df['planet_star_radius_ratio']
    arg1_sqrt = ((1 + p)**2 - b**2).clip(0)
    arg1 = (np.sqrt(arg1_sqrt) / df['sma']).clip(-1, 1)
    df['transit_duration_est'] = (df['P'] / np.pi) * np.arcsin(arg1)
    arg2_sqrt = ((1 - p)**2 - b**2).clip(0)
    arg2 = (np.sqrt(arg2_sqrt) / df['sma']).clip(-1, 1)
    T_flat_est = (df['P'] / np.pi) * np.arcsin(arg2)
    df['ingress_egress_duration_est'] = (df['transit_duration_est'] - T_flat_est) / 2.0
    df['ingress_duration_ratio'] = (df['ingress_egress_duration_est'] / df['transit_duration_est']).fillna(0)
    df['planet_eq_temp_est'] = df['Ts'] * np.sqrt(1 / (2 * df['sma']))
    df['planet_gravity_proxy'] = df['Mp'] / (p * df['Rs'])**2
    df['eclipsed_light_proxy'] = df['fgs_depth_mean'] * (df['Rs']**2)
    df['duration_period_ratio'] = df['transit_duration_est'] / df['P']
    print(f"Added {8} new physics-based features.")
    return df

def extract_timeseries_features(f_raw, a_raw):
    features = {}
    fgs_transit_slice = slice(23500, 44000)
    fgs_out_of_transit_mask = np.ones(len(f_raw), dtype=bool); fgs_out_of_transit_mask[fgs_transit_slice] = False
    fgs_unobscured_mean = np.mean(f_raw[fgs_out_of_transit_mask])
    fgs_transit = f_raw[fgs_transit_slice]
    fgs_depth = (fgs_unobscured_mean - np.mean(fgs_transit)) / fgs_unobscured_mean
    features['fgs_depth'] = fgs_depth
    for i in range(5):
        features[f'fgs_slice_{i+1}'] = (fgs_unobscured_mean - np.mean(fgs_transit[i*4100:(i+1)*4100])) / fgs_unobscured_mean
    features['fgs_transit_std'] = fgs_transit.std()
    features['fgs_snr'] = fgs_depth / (fgs_transit.std() + 1e-6)
    features.update({f'fgs_{k}': v for k, v in get_detrended_features(f_raw, fgs_transit_slice).items()})
    features['fgs_noise_autocorr'] = autocorrelation(f_raw[fgs_out_of_transit_mask])
    airs_transit_slice = slice(1950, 3700)
    airs_out_of_transit_mask = np.ones(len(a_raw), dtype=bool); airs_out_of_transit_mask[airs_transit_slice] = False
    airs_unobscured_mean = np.mean(a_raw[airs_out_of_transit_mask])
    airs_transit = a_raw[airs_transit_slice]
    airs_depth = (airs_unobscured_mean - np.mean(airs_transit)) / airs_unobscured_mean
    features['airs_depth'] = airs_depth
    slice_len = len(airs_transit) // 5
    for i in range(5):
        features[f'airs_slice_{i+1}'] = (airs_unobscured_mean - np.mean(airs_transit[i*slice_len:(i+1)*slice_len])) / airs_unobscured_mean
    features['airs_transit_std'] = airs_transit.std()
    features['airs_snr'] = airs_depth / (airs_transit.std() + 1e-6)
    features.update({f'airs_{k}': v for k, v in get_detrended_features(a_raw, airs_transit_slice).items()})
    features['airs_noise_autocorr'] = autocorrelation(a_raw[airs_out_of_transit_mask])
    features['depth_ratio_fgs_airs'] = fgs_depth / (airs_depth + 1e-6)
    return features

def combined_feature_engineering(fgs_signals, airs_signals, star_info_df):
    print("Engineering and Aggregating features...")
    all_planet_features_list = []
    for planet_id in tqdm(star_info_df.index, desc="Feature Engineering"):
        num_obs = len(fgs_signals.get(planet_id, []))
        if num_obs == 0: continue
        obs_features_list = [extract_timeseries_features(fgs_signals[planet_id][i], airs_signals[planet_id][i]) for i in range(num_obs)]
        obs_features_df = pd.DataFrame(obs_features_list)
        aggregated_feats = {f'{col}_{agg}': obs_features_df[col].agg(agg) for col in obs_features_df.columns for agg in ['mean', 'std', 'min', 'max']}
        aggregated_feats['planet_id'] = planet_id
        aggregated_feats['num_observations'] = num_obs
        all_planet_features_list.append(aggregated_feats)
    
    features_df = pd.DataFrame(all_planet_features_list).set_index('planet_id')
    meta_df = star_info_df.copy().fillna(star_info_df.median())
    meta_df['impact_parameter_b'] = meta_df['sma'] * np.cos(np.deg2rad(meta_df['i']))
    meta_df['rho_star_proxy'] = meta_df['Ms'] / (meta_df['Rs']**3)
    
    poly = PolynomialFeatures(degree=2, include_bias=False, interaction_only=True)
    poly_cols = ['Rs', 'Ts', 'Mp', 'P', 'impact_parameter_b', 'rho_star_proxy']
    poly_features = poly.fit_transform(meta_df[poly_cols])
    poly_df = pd.DataFrame(poly_features, columns=poly.get_feature_names_out(poly_cols), index=meta_df.index)
    
    final_features_df = pd.concat([features_df, meta_df, poly_df], axis=1).loc[:, ~pd.concat([features_df, meta_df, poly_df], axis=1).columns.duplicated()]
    final_features_df = add_physics_and_interaction_features(final_features_df)
    
    print(f"Created {final_features_df.shape[1]} features in total.")
    return final_features_df.fillna(0).replace([np.inf, -np.inf], 0)

def official_competition_score(y_true, y_pred, sigma_pred, naive_mean, naive_sigma, fsg_sigma_true=1e-6, airs_sigma_true=1e-5):
    y_true, y_pred, sigma_pred = np.asarray(y_true), np.asarray(y_pred), np.asarray(sigma_pred)
    sigma_pred = np.clip(sigma_pred, 1e-15, None)
    n_wavelengths = y_true.shape[1]
    
    if n_wavelengths == 283:
        sigma_true_arr = np.append(np.array([fsg_sigma_true]), np.ones(n_wavelengths - 1) * airs_sigma_true)
        FGS1_RELATIVE_WEIGHT = 57.846
        weights_arr = np.append(np.array([FGS1_RELATIVE_WEIGHT]), np.ones(n_wavelengths - 1))
    elif n_wavelengths == 1:
        sigma_true_arr = np.array([fsg_sigma_true])
        weights_arr = np.ones(n_wavelengths)
    else:
        sigma_true_arr = np.ones(n_wavelengths) * airs_sigma_true
        weights_arr = np.ones(n_wavelengths)

    sigma_true = np.tile(sigma_true_arr, (y_true.shape[0], 1))
    weights = np.tile(weights_arr, (y_true.shape[0], 1))
    
    gll_pred = scipy.stats.norm.logpdf(y_true, loc=y_pred, scale=sigma_pred)
    gll_true = scipy.stats.norm.logpdf(y_true, loc=y_true, scale=sigma_true)
    gll_mean = scipy.stats.norm.logpdf(y_true, loc=naive_mean, scale=naive_sigma)
    ind_scores = (gll_pred - gll_mean) / (gll_true - gll_mean + 1e-9)
    final_score = np.average(ind_scores, weights=weights)
    return float(np.clip(final_score, 0.0, 1.0))

def find_best_calibration(y_true, y_pred, sigma_raw, naive_mean, naive_sigma, instrument_name):
    print(f"\n--- [Validation] Searching for best calibration factors for {instrument_name}... ---")
    best_score, best_scaling, best_additive = -1.0, 1.0, 0.0
    sigma_true_const = 1e-6 if "FGS" in instrument_name else 1e-5
    
    for scaling in config.CALIBRATION_SCALING_FACTORS:
        for additive in config.CALIBRATION_ADDITIVE_FACTORS:
            sigma_calibrated = (sigma_raw * scaling) + additive
            score = official_competition_score(y_true, y_pred, sigma_calibrated, naive_mean, naive_sigma,
                                               fsg_sigma_true=sigma_true_const, airs_sigma_true=sigma_true_const)
            if score > best_score:
                best_score, best_scaling, best_additive = score, scaling, additive
    
    print(f"--- Best factors for {instrument_name}: Scale={best_scaling}, Add={best_additive} (Best Score: {best_score:.4f}) ---")
    return {'scaling': best_scaling, 'additive': best_additive}
# --------------------------------------------------------------------------------------------------------------

if __name__ == '__main__':
    ## ======================================================================
    ## --- SUBMISSION MODE ---
    ## ======================================================================
    if not TRAIN:
        print("\n" + "="*50 + "\n======         SUBMISSION MODE          ======\n" + "="*50 + "\n")
        
        sample_submission = pd.read_csv(config.SAMPLE_SUBMISSION_PATH, index_col='planet_id')
        test_star_info_df = pd.read_csv(config.TEST_STAR_INFO_PATH, index_col='planet_id')
        
        fgs_signals_test = load_all_observations('test', test_star_info_df.index, 'FGS1')
        airs_signals_test = load_all_observations('test', test_star_info_df.index, 'AIRS-CH0')
        test_features_df = combined_feature_engineering(fgs_signals_test, airs_signals_test, test_star_info_df)
        
        print("Loading saved models and parameters...")
        with open(os.path.join(config.OUTPUT_PATH, 'feature_columns.pkl'), 'rb') as f: train_cols = pickle.load(f)
        with open(os.path.join(config.OUTPUT_PATH, 'calibration_params.pkl'), 'rb') as f: calibration_params = pickle.load(f)

        print("Aligning test features with training features...")
        test_features_df = test_features_df.reindex(columns=train_cols).fillna(0)

        # --- Predict using the ensemble of models ---
        all_mu_preds = []
        all_sigma_preds = []
        for fold in range(config.N_FOLDS):
            print(f"--- Predicting with models from Fold {fold+1}/{config.N_FOLDS} ---")
            with open(os.path.join(config.OUTPUT_PATH, f'model_mean_fold_{fold}.pkl'), 'rb') as f: model_mean = pickle.load(f)
            with open(os.path.join(config.OUTPUT_PATH, f'model_sigma_fold_{fold}.pkl'), 'rb') as f: model_sigma = pickle.load(f)
            
            mu_pred = model_mean.predict(test_features_df)
            sigma_pred = model_sigma.predict(test_features_df)
            
            all_mu_preds.append(mu_pred)
            all_sigma_preds.append(sigma_pred)
            
        # --- Average the predictions ---
        print("Averaging predictions from all folds...")
        y_pred_test = np.mean(all_mu_preds, axis=0).clip(0, None)
        sigma_raw_test = np.mean(all_sigma_preds, axis=0).clip(1e-10, None)

        # --- Apply the single, globally optimized calibration ---
        sigma_pred_test_fgs1 = (sigma_raw_test[:, :1] * calibration_params['fgs1']['scaling']) + calibration_params['fgs1']['additive']
        sigma_pred_test_airs = (sigma_raw_test[:, 1:] * calibration_params['airs']['scaling']) + calibration_params['airs']['additive']
        sigma_pred_test = np.hstack([sigma_pred_test_fgs1, sigma_pred_test_airs])

        print("Creating submission file...")
        pred_df = pd.DataFrame(y_pred_test, index=sample_submission.index, columns=sample_submission.columns[:283])
        sigma_df = pd.DataFrame(sigma_pred_test, index=sample_submission.index, columns=sample_submission.columns[283:])
        submission_df = pd.concat([pred_df, sigma_df], axis=1)
        
        submission_df.to_csv('submission_xgb.csv')
        print(submission_df.head())
        print("\n'submission.csv' created successfully!")

    ## ======================================================================
    ## --- TRAINING MODE ---
    ## ======================================================================
    else:
        print("\n" + "="*50 + "\n======          TRAINING MODE           ======\n" + "="*50 + "\n")
        
        os.makedirs(config.OUTPUT_PATH, exist_ok=True)
        
        train_labels_df = pd.read_csv(config.TRAIN_LABELS_PATH, index_col='planet_id')
        train_star_info_df = pd.read_csv(config.TRAIN_STAR_INFO_PATH, index_col='planet_id').loc[train_labels_df.index]
        
        if DEBUG:
            train_labels_df = train_labels_df.head(100)
            train_star_info_df = train_star_info_df.head(100)

        fgs_signals_train = load_all_observations('train', train_labels_df.index, 'FGS1')
        airs_signals_train = load_all_observations('train', train_labels_df.index, 'AIRS-CH0')
        
        train_features_df = combined_feature_engineering(fgs_signals_train, airs_signals_train, train_star_info_df)
        train_labels = train_labels_df.reindex(train_features_df.index).values
        naive_mu_train, naive_sigma_train = np.mean(train_labels), np.std(train_labels)

        # --- Save feature columns before training ---
        with open(os.path.join(config.OUTPUT_PATH, 'feature_columns.pkl'), 'wb') as f:
            pickle.dump(train_features_df.columns.tolist(), f)
            
        # --- Prepare for K-Fold training ---
        # Stratify on a key feature. Binning continuous values is good practice for stratification.
        stratify_col = pd.cut(train_features_df['fgs_depth_mean'], bins=10, labels=False)
        skf = StratifiedKFold(n_splits=config.N_FOLDS, shuffle=True, random_state=config.RANDOM_STATE)
        
        oof_mu = np.zeros_like(train_labels)
        oof_sigma = np.zeros_like(train_labels)
        
        for fold, (train_idx, val_idx) in enumerate(skf.split(train_features_df, stratify_col)):
            print("\n" + "*"*20 + f" FOLD {fold+1}/{config.N_FOLDS} " + "*"*20)
            
            # --- Correctly split data using indices ---
            X_train_f, X_val_f = train_features_df.iloc[train_idx], train_features_df.iloc[val_idx]
            y_train_f, y_val_f = train_labels[train_idx], train_labels[val_idx]

            # --- Stage 1: Train Mean Model for this fold ---
            print(f"--- [Fold {fold+1}] Training Mean (mu) Model... ---")
            model_mean = MultiOutputRegressor(xgb.XGBRegressor(**config.XGB_PARAMS_MEAN), n_jobs=-1)
            model_mean.fit(X_train_f, y_train_f)

            # --- Stage 2: Train Sigma Model for this fold ---
            print(f"--- [Fold {fold+1}] Training Sigma (uncertainty) Model... ---")
            y_pred_mu_val = model_mean.predict(X_val_f)
            y_target_sigma_val = np.abs(y_val_f - y_pred_mu_val)
            
            model_sigma = MultiOutputRegressor(xgb.XGBRegressor(**config.XGB_PARAMS_SIGMA), n_jobs=-1)
            model_sigma.fit(X_val_f, y_target_sigma_val)

            # --- Store Out-of-Fold (OOF) predictions for final validation ---
            oof_mu[val_idx] = y_pred_mu_val
            oof_sigma[val_idx] = model_sigma.predict(X_val_f)
            
            # --- Stage 3: Save the models for this fold ---
            print(f"--- [Fold {fold+1}] Saving models... ---")
            with open(os.path.join(config.OUTPUT_PATH, f'model_mean_fold_{fold}.pkl'), 'wb') as f:
                pickle.dump(model_mean, f)
            with open(os.path.join(config.OUTPUT_PATH, f'model_sigma_fold_{fold}.pkl'), 'wb') as f:
                pickle.dump(model_sigma, f)
        
        # --- Stage 4: Validate and Calibrate using ALL OOF predictions ---
        print("\n" + "="*50 + "\n======   FINAL OOF VALIDATION & CALIBRATION   ======\n" + "="*50 + "\n")
        
        # Clip OOF predictions
        oof_mu_clipped = oof_mu.clip(0, None)
        oof_sigma_clipped = oof_sigma.clip(1e-10, None)

        # Split OOF predictions by instrument for separate calibration
        y_true_fgs1, y_true_airs = train_labels[:, :1], train_labels[:, 1:]
        oof_mu_fgs1, oof_mu_airs = oof_mu_clipped[:, :1], oof_mu_clipped[:, 1:]
        oof_sigma_fgs1, oof_sigma_airs = oof_sigma_clipped[:, :1], oof_sigma_clipped[:, 1:]
        
        best_params_fgs1 = find_best_calibration(y_true_fgs1, oof_mu_fgs1, oof_sigma_fgs1, naive_mu_train, naive_sigma_train, "FGS1")
        best_params_airs = find_best_calibration(y_true_airs, oof_mu_airs, oof_sigma_airs, naive_mu_train, naive_sigma_train, "AIRS")
        
        # Apply best calibration to OOF sigma to calculate final score
        oof_sigma_calibrated_fgs1 = (oof_sigma_fgs1 * best_params_fgs1['scaling']) + best_params_fgs1['additive']
        oof_sigma_calibrated_airs = (oof_sigma_airs * best_params_airs['scaling']) + best_params_airs['additive']
        oof_sigma_calibrated_total = np.hstack([oof_sigma_calibrated_fgs1, oof_sigma_calibrated_airs])
        
        final_cv_score = official_competition_score(train_labels, oof_mu_clipped, oof_sigma_calibrated_total, naive_mu_train, naive_sigma_train)
        print(f"\n--- Final Combined OOF CV Score: {final_cv_score:.5f} ---")
        
        # --- Stage 5: Save the final calibration parameters ---
        calibration_params = {'fgs1': best_params_fgs1, 'airs': best_params_airs}
        with open(os.path.join(config.OUTPUT_PATH, 'calibration_params.pkl'), 'wb') as f:
            pickle.dump(calibration_params, f)
            
        print("\nTraining complete. All models and artifacts saved to:", config.OUTPUT_PATH)



======         SUBMISSION MODE          ======

Loading ALL observations for FGS1 in test set...


Dispatching FGS1 jobs: 100%|██████████| 1/1 [00:00<00:00, 123.89it/s]


Loading ALL observations for AIRS-CH0 in test set...


Dispatching AIRS-CH0 jobs: 100%|██████████| 1/1 [00:00<00:00, 532.34it/s]


Engineering and Aggregating features...


Feature Engineering: 100%|██████████| 1/1 [00:00<00:00, 17.82it/s]

Adding physics-informed and interaction features...
Added 8 new physics-based features.
Created 134 features in total.
Loading saved models and parameters...
Aligning test features with training features...
--- Predicting with models from Fold 1/5 ---



/usr/local/lib/python3.11/dist-packages/xgboost/core.py:160: UserWarning: [05:33:50] WARNING: /workspace/src/common/error_msg.h:80: If you are loading a serialized model (like pickle in Python, RDS in R) or
configuration generated by an older version of XGBoost, please export the model by calling
`Booster.save_model` from that version first, then load it back in current version. See:

    https://xgboost.readthedocs.io/en/stable/tutorials/saving_model.html

for more details about differences between saving model and serializing.

  warnings.warn(smsg, UserWarning)
/usr/local/lib/python3.11/dist-packages/sklearn/base.py:318: UserWarning: Trying to unpickle estimator MultiOutputRegressor from version 1.7.0 when using version 1.2.2. This might lead to breaking code or invalid results. Use at your own risk. For more info please refer to:
https://scikit-learn.org/stable/model_persistence.html#security-maintainability-limitations
  warnings.warn(


--- Predicting with models from Fold 2/5 ---
--- Predicting with models from Fold 3/5 ---
--- Predicting with models from Fold 4/5 ---
--- Predicting with models from Fold 5/5 ---
Averaging predictions from all folds...
Creating submission file...
               wl_1      wl_2      wl_3      wl_4      wl_5      wl_6  \
planet_id                                                               
1103775    0.015921  0.016237  0.016225  0.016209  0.016204  0.016207   

               wl_7      wl_8      wl_9     wl_10  ...  sigma_274  sigma_275  \
planet_id                                          ...                         
1103775    0.016196  0.016219  0.016203  0.016183  ...   0.000389   0.000381   

           sigma_276  sigma_277  sigma_278  sigma_279  sigma_280  sigma_281  \
planet_id                                                                     
1103775     0.000386   0.000381   0.000387   0.000391   0.000381   0.000386   

           sigma_282  sigma_283  
planet_id          

In [24]:
xgb = pd.read_csv("submission_xgb.csv")
nn = pd.read_csv("submission_nn.csv")

In [25]:
# Identify columns
wl_cols    = [c for c in xgb.columns if c.startswith("wl_")]
sigma_cols = [c for c in xgb.columns if c.startswith("sigma_")]

# Weighted blends
wl_blend    = 0.2 * xgb[wl_cols] + 0.8 * nn[wl_cols]
sigma_blend = 0.4 * xgb[sigma_cols] + 0.6 * nn[sigma_cols]

# Combine with planet_id
submission = pd.concat(
    [xgb[["planet_id"]], wl_blend, sigma_blend],
    axis=1
)

# Save final submission
submission.to_csv("submission_396.csv", index=False)
print("Saved submission.csv with shape:", submission.shape)

Saved submission.csv with shape: (1, 567)


In [26]:
import pandas as pd

h = pd.read_csv("submission_396.csv")
l = pd.read_csv("submission_380.csv")

wl_cols    = [c for c in h.columns if c.startswith("wl_")]
sigma_cols = [c for c in h.columns if c.startswith("sigma_")]

# choose a heavier weight for the better file
w_h = 0.75     # weight for high-score file
w_l = 0.25     # weight for lower-score file

blend = h.copy()
blend[wl_cols]    = w_h * h[wl_cols]    + w_l * l[wl_cols]
blend[sigma_cols] = w_h * h[sigma_cols] + w_l * l[sigma_cols]

blend.to_csv("submission.csv", index=False)


In [27]:
blend

,planet_id,wl_1,wl_2,wl_3,wl_4,wl_5,wl_6,wl_7,wl_8,wl_9,...,sigma_274,sigma_275,sigma_276,sigma_277,sigma_278,sigma_279,sigma_280,sigma_281,sigma_282,sigma_283
0,1103775,0.016028,0.015736,0.015759,0.015737,0.015762,0.015738,0.015798,0.015762,0.015783,...,0.000507,0.000504,0.000506,0.000504,0.000506,0.000507,0.000504,0.000506,0.00051,0.000506
